In [1]:
%load_ext autoreload
%autoreload 2

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
import sys, os, re
from pathlib import Path
from typing import Optional
sys.path.append('../')

from src import *

seed = 42
np.random.seed(seed)
warnings.simplefilter(action='ignore', category=FutureWarning)

In [15]:
df_info = create_metadata_dataframe('../data')
df_info

,path,label,subdir,region,center,place
0,../data/control/mk2a/striatum_left_control_2Ag...,0,mk2a,striatum,2900,2
1,../data/control/mk2a/cortex_left_control_2Agro...,0,mk2a,cortex,1500,2
2,../data/control/mk2a/cortex_right_control_2Agr...,0,mk2a,cortex,1500,3
3,../data/control/mk2a/striatum_right_control_2A...,0,mk2a,striatum,2900,3
4,../data/control/mk2a/striatum_left_control_2Ag...,0,mk2a,striatum,2900,1
...,...,...,...,...,...,...
234,../data/exo/mexo3/cerebellum_right_exo_3group_...,2,mexo3,cerebellum,1500,3
235,../data/exo/mexo3/cerebellum_right_exo_3group_...,2,mexo3,cerebellum,1500,1
236,../data/exo/mexo3/cerebellum_left_exo_3group_6...,2,mexo3,cerebellum,1500,1
237,../data/exo/mexo3/cerebellum_left_exo_3group_6...,2,mexo3,cerebellum,2900,1


In [16]:
def load_spectrum(file_path):
    df = pd.read_csv(file_path, sep='\t')  # или ' ' если пробелы; проверьте header=None если нет заголовков
    wavenumbers = df.iloc[:, 0].values
    spectra = df.iloc[:, 1:].values.T  # (525, num_wavenumbers)
    return wavenumbers, spectra

In [ ]:
load_spectrum()

In [2]:
import os
import re
import numpy as np
import pandas as pd
from pathlib import Path
from concurrent.futures import ProcessPoolExecutor, as_completed
from functools import partial

# ─── helpers ────────────────────────────────────────────────────────────────

_GROUP_RE = re.compile(r'(?:mk|mend|mexo)(\d[ab]?)', re.IGNORECASE)
_CENTER_RE = re.compile(r'center(\d+)')
_BRAIN_RE  = re.compile(
    r'^(cerebellum_left|cerebellum_right|'
    r'striatum_left|striatum_right|'
    r'cortex_left|cortex_right|cortex|'
    r'striatum)'
)

def _parse_meta(filepath: Path) -> dict:
    """Extract metadata from path components."""
    parts = filepath.parts          # …/control/mk2a/filename.txt
    
    # label  ──  top-level class dir
    # find 'control'/'endo'/'exo' in path
    label = next((p for p in parts if p in ('control', 'endo', 'exo')), None)

    # group  ──  sub-dir name  (mk1 → "1",  mend2a → "2a",  mexo2b → "2b")
    subdir = parts[-2]              # directory containing the file
    m = _GROUP_RE.search(subdir)
    group = m.group(1).lower() if m else None

    name = filepath.stem            # filename without .txt

    # center frequency
    m = _CENTER_RE.search(name)
    center = int(m.group(1)) if m else None

    # brain region  ──  prefix before the first class keyword
    m = _BRAIN_RE.match(name)
    brain = m.group(1) if m else None

    return dict(label=label, group=group, center=center, brain=brain,
                filepath=str(filepath))


def _read_one(filepath_str: str) -> pd.DataFrame | None:
    """Read a single spectrum file; return a 1-row DataFrame or None on error."""
    fp = Path(filepath_str)
    try:
        # skip comment lines that start with '#', read only Wave & Intensity cols
        arr = np.loadtxt(fp, comments='#', usecols=(2, 3))   # Wave, Intensity
        if arr.ndim == 1:
            arr = arr[np.newaxis, :]
    except Exception:
        return None

    meta = _parse_meta(fp)

    # store spectrum as two numpy arrays packed in a single row
    row = {**meta,
           'wavenumber': arr[:, 0],
           'intensity':  arr[:, 1]}
    return row          # just a dict – cheaper to build DF later


# ─── public API ─────────────────────────────────────────────────────────────

def load_all_spectra(root: str | Path,
                     n_workers: int = os.cpu_count(),
                     skip_average: bool = True) -> pd.DataFrame:
    """
    Recursively load every .txt spectrum under *root*.

    Parameters
    ----------
    root        : path to the dataset root (contains control/, endo/, exo/)
    n_workers   : parallel worker processes (default = all CPUs)
    skip_average: skip files whose name contains '_Average' (default True)

    Returns
    -------
    pd.DataFrame with columns:
        label, group, center, brain, filepath, wavenumber, intensity
    where `wavenumber` and `intensity` are numpy arrays (one spectrum per row).
    """
    root = Path(root)
    files = [
        str(p) for p in root.rglob('*.txt')
        if not (skip_average and '_Average' in p.name)
    ]
    print(f"Found {len(files)} spectrum files – loading with {n_workers} workers …")

    rows = []
    with ProcessPoolExecutor(max_workers=n_workers) as pool:
        futures = {pool.submit(_read_one, f): f for f in files}
        for fut in as_completed(futures):
            result = fut.result()
            if result is not None:
                rows.append(result)

    df = pd.DataFrame(rows)

    # tidy dtypes
    df['label']  = df['label'].astype('category')
    df['group']  = df['group'].astype('category')
    df['brain']  = df['brain'].astype('category')
    df['center'] = df['center'].astype('Int16')

    print(f"Loaded {len(df)} spectra. Shape: {df.shape}")
    print(df[['label','group','center','brain']].value_counts().to_string())
    return df

In [3]:
df = load_all_spectra('../data')

Found 237 spectrum files – loading with 16 workers …
Loaded 237 spectra. Shape: (237, 7)
label    group  center  brain           
endo     1      2900    cortex              7
exo      1      2900    cortex              6
                1500    cortex              6
         3      2900    cerebellum_right    6
                        cerebellum_left     6
                1500    cerebellum_right    6
                        cerebellum_left     6
endo     1      1500    cortex              6
control  1      1500    cortex              6
                2900    cortex              6
         3      2900    cerebellum_right    5
                        cerebellum_left     5
                1500    cerebellum_left     5
                        cerebellum_right    5
         2a     1500    cortex_left         4
                2900    cortex_left         4
         2b     2900    cortex_left         4
                1500    cortex_left         4
exo      2b     2900    striatum_right    

In [4]:
df

,label,group,center,brain,filepath,wavenumber,intensity
0,exo,2b,2900,striatum_right,../data/exo/mexo2b/striatum_right_exo_2Bgroup_...,"[3288.228516, 3287.485352, 3286.741211, 3285.9...","[4316.587891, 4269.30127, 4285.396484, 4271.09..."
1,exo,2b,2900,striatum_left,../data/exo/mexo2b/striatum_left_exo_2Bgroup_6...,"[3288.231445, 3287.488281, 3286.744141, 3286.0...","[6549.193359, 6586.347168, 6436.541016, 6621.0..."
2,exo,2b,2900,cortex_left,../data/exo/mexo2b/cortex_left_exo_2Bgroup_633...,"[3288.231445, 3287.488281, 3286.744141, 3286.0...","[6397.200684, 6481.641113, 6406.147949, 6205.7..."
3,exo,2b,2900,striatum_left,../data/exo/mexo2b/striatum_left_exo_2Bgroup_6...,"[3288.228516, 3287.485352, 3286.741211, 3285.9...","[9852.493164, 9629.571289, 9708.836914, 9588.8..."
4,exo,2b,2900,cortex_left,../data/exo/mexo2b/cortex_left_exo_2Bgroup_633...,"[3288.154297, 3287.411133, 3286.667969, 3285.9...","[4708.312012, 4684.669434, 4683.802734, 4635.6..."
...,...,...,...,...,...,...,...
232,control,1,2900,cortex,../data/control/mk1/cortex_control_1group_633n...,"[3288.18457, 3287.441406, 3286.698242, 3285.95...","[5029.213867, 4992.060547, 4940.48291, 4939.56..."
233,control,1,2900,cortex,../data/control/mk1/cortex_control_1group_633n...,"[3288.18457, 3287.441406, 3286.698242, 3285.95...","[6238.386719, 6251.896973, 6426.342285, 6330.6..."
234,control,1,1500,cortex,../data/control/mk1/cortex_control_1group_633n...,"[2002.417969, 2001.458008, 2000.498047, 1999.5...","[8813.924805, 8821.768555, 8822.693359, 8834.0..."
235,control,1,1500,cortex,../data/control/mk1/cortex_control_1group_633n...,"[2002.299805, 2001.339844, 2000.379883, 1999.4...","[15454.765625, 15256.057617, 15595.579102, 153..."


In [5]:
flat = df.explode(['wavenumber', 'intensity']).reset_index(drop=True)
flat['wavenumber'] = flat['wavenumber'].astype('float32')
flat['intensity']  = flat['intensity'].astype('float32')
flat

KeyboardInterrupt: 